# ClaimsIQ 04 — SWARM Fraud Triage, Connected to Snowflake via MCP

**This notebook:** Rebuilds Lab 20's Triage → Fraud Specialist →
Escalation handoff chain, now pulling real data from Snowflake through
the SAME `mcp_snowflake_server.py` module as Notebooks 02-03.

### Prerequisite
Run `00_snowflake_setup_and_seed_data.ipynb` and
`01_mcp_server_snowflake.ipynb` first, in this Jupyter environment.

## Step 1 — Install & import

In [ ]:
%pip install -q openai snowflake-connector-python

In [ ]:
import os, json
from openai import OpenAI
from mcp_snowflake_server import claims_server, SimpleMCPClient

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before continuing"
client = OpenAI()

mcp_client = SimpleMCPClient(claims_server)
mcp_client.connect()
print("SWARM agent connected to the Snowflake MCP server.")

## Step 2 — Agent instructions

Same three roles as Lab 20, with thresholds re-tuned for real customer
data and a fixed arithmetic issue from Lab 20's troubleshooting: the
ratio is computed by the TOOL, not the model, so the Fraud Specialist
never has to do division in its head.

In [ ]:
TRIAGE_INSTRUCTIONS = """You are the Triage Agent for ShopEase fraud detection.
Call get_transaction_velocity for the customer. If unrecognized_device_txns
is 2 or more, call transfer_to_fraud_specialist. Otherwise conclude the
account is normal and approve directly."""

FRAUD_SPECIALIST_INSTRUCTIONS = """You are the Fraud Specialist. You receive
accounts already flagged by Triage. Call check_fraud_signals and
compare_to_peer_spend. Look at the customer_to_peer_ratio field returned —
do not calculate it yourself, use the value given.

- If customer_to_peer_ratio is above 3 AND there is a HIGH severity fraud
  signal, decide DIRECTLY: high risk, recommend deeper investigation.
- If customer_to_peer_ratio is below 1.5, decide DIRECTLY: low risk, likely
  a market-wide trend (e.g. a sale event), not personal fraud.
- For every OTHER value, you MUST call transfer_to_escalation."""

ESCALATION_INSTRUCTIONS = """You are the Escalation Agent. You receive
ambiguous fraud cases. Call get_customer_profile for context (tenure, risk
tier). Summarize the case clearly and state it has been flagged for human
review — do not make a final risk decision yourself."""

print("Instructions defined.")

## Step 3 — Tools, including handoffs

Same structure as Lab 20 — real tools call `mcp_client.call_tool(...)`,
handoff tools swap the active agent.

In [ ]:
def get_transaction_velocity(customer_id: str) -> dict:
    return mcp_client.call_tool("get_transaction_velocity", customer_id=customer_id)

def check_fraud_signals(customer_id: str) -> list:
    return mcp_client.call_tool("check_fraud_signals", customer_id=customer_id)

def compare_to_peer_spend(customer_id: str) -> dict:
    return mcp_client.call_tool("compare_to_peer_spend", customer_id=customer_id)

def get_customer_profile(customer_id: str) -> dict:
    return mcp_client.call_tool("get_customer_profile", customer_id=customer_id)

TOOL_FUNCTIONS = {
    "get_transaction_velocity": get_transaction_velocity,
    "check_fraud_signals": check_fraud_signals,
    "compare_to_peer_spend": compare_to_peer_spend,
    "get_customer_profile": get_customer_profile,
}

tools = [
    {"type": "function", "function": {"name": "get_transaction_velocity", "description": "Returns transaction count/total in last 48h and unrecognized device count.", "parameters": {"type": "object", "properties": {"customer_id": {"type": "string"}}, "required": ["customer_id"]}}},
    {"type": "function", "function": {"name": "check_fraud_signals", "description": "Checks fraud signals for a customer in the last 30 days.", "parameters": {"type": "object", "properties": {"customer_id": {"type": "string"}}, "required": ["customer_id"]}}},
    {"type": "function", "function": {"name": "compare_to_peer_spend", "description": "Compares customer average spend to peer average, returns customer_to_peer_ratio.", "parameters": {"type": "object", "properties": {"customer_id": {"type": "string"}}, "required": ["customer_id"]}}},
    {"type": "function", "function": {"name": "get_customer_profile", "description": "Looks up customer profile and recent orders.", "parameters": {"type": "object", "properties": {"customer_id": {"type": "string"}}, "required": ["customer_id"]}}},
    {"type": "function", "function": {"name": "transfer_to_fraud_specialist", "description": "Hands off to the Fraud Specialist for deeper investigation.", "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {"name": "transfer_to_escalation", "description": "Hands off to Escalation for human review when genuinely ambiguous.", "parameters": {"type": "object", "properties": {}}}},
]

HANDOFF_TOOLS = {"transfer_to_fraud_specialist": "fraud_specialist", "transfer_to_escalation": "escalation"}
AGENT_INSTRUCTIONS = {"triage": TRIAGE_INSTRUCTIONS, "fraud_specialist": FRAUD_SPECIALIST_INSTRUCTIONS, "escalation": ESCALATION_INSTRUCTIONS}
print("Tools and handoff map ready.")

## Step 4 — The run loop (identical structure to Lab 20)

In [ ]:
def run_swarm(customer_id, max_hops=6, verbose=True):
    current_agent = "triage"
    handoff_chain = [current_agent]
    messages = [
        {"role": "system", "content": AGENT_INSTRUCTIONS[current_agent]},
        {"role": "user", "content": f"Evaluate fraud risk for customer {customer_id}."},
    ]

    for hop in range(max_hops):
        response = client.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools, temperature=0)
        message = response.choices[0].message

        if not message.tool_calls:
            if verbose:
                print(f"[{current_agent}] Final: {message.content}")
            return message.content, handoff_chain

        messages.append(message)
        for tc in message.tool_calls:
            args = json.loads(tc.function.arguments)
            name = tc.function.name

            if name in HANDOFF_TOOLS:
                new_agent = HANDOFF_TOOLS[name]
                if verbose:
                    print(f"[{current_agent}] -> handoff -> [{new_agent}]")
                current_agent = new_agent
                handoff_chain.append(current_agent)
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": f"Transferred to {new_agent}."})
                messages[0] = {"role": "system", "content": AGENT_INSTRUCTIONS[current_agent]}
            else:
                result = TOOL_FUNCTIONS[name](**args)
                if verbose:
                    print(f"[{current_agent}] called {name}({args}) -> {result}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result, default=str)})

    return "Max hops reached.", handoff_chain

print("run_swarm() ready.")

## Step 5 — Run Ananya Rao's case

Note: this notebook's Triage checks TRANSACTION VELOCITY (unrecognized
devices), unlike Notebooks 02-03 which start from the claim itself —
SWARM here is framed as an account-level fraud check, a realistic
parallel process that could run independently of any specific claim.

In [ ]:
answer, chain = run_swarm("CUST99001")
print("\nHandoff chain:", " -> ".join(chain))

## Step 6 — Run a clearly normal customer for contrast

Pick any customer ID from Notebook 00's synthetic population that
ISN'T Ananya Rao — confirm Triage handles it without escalating.

In [ ]:
answer2, chain2 = run_swarm("CUST00010")
print("\nHandoff chain:", " -> ".join(chain2))

## Deliverable

1. Both handoff chains and final answers.
2. Did Ananya Rao's case escalate, resolve directly at Fraud Specialist,
   or get approved at Triage? Compare this outcome to Notebooks 02 and
   03's decisions on the same underlying account — do all three agree
   on the LEVEL of concern, even though SWARM here only assesses account
   risk, not the specific claim?
3. One paragraph: SWARM's Triage only ever looks at
   `unrecognized_device_txns` to decide whether to hand off. What real
   fraud case would this miss that Notebook 02's LangGraph agent (which
   pulls profile, claim, AND fraud data together before any decision)
   would catch?